In [1]:
from pathlib import Path
from PIL import Image

In [2]:
# Gather input images
in_dir = Path("./unprocessed")
out_dir = Path("./processed")

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
filenames = sorted(
    p.name for p in in_dir.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
)

In [ ]:
# Threshold filter, picked by eyeballing it
THRESH = 160

for name in filenames:
    with Image.open(in_dir / name) as img:
        bw = img.convert("L").point(lambda p: 255 if p > THRESH else 0)
        bw.save(out_dir / name)  # same name, different folder

In [ ]:
# Step 2: straighten the black-and-white images with the Hough line transform
import cv2
import numpy as np

(out_dir / "straightened").mkdir(exist_ok=True)

angles = {}
for name in filenames:
    img = cv2.imread(str(out_dir / name), cv2.IMREAD_GRAYSCALE)   # thresholded image
    h, w = img.shape

    # 1. Find edges, then long straight lines (at least 1/10 of the image width)
    edges = cv2.Canny(img, 50, 150)
    lines = cv2.HoughLines(edges, 1, np.pi / 1800, w // 10)       # angle step 0.1°
    theta = np.empty(0) if lines is None else lines.reshape(-1, 2)[:, 1]

    # 2. How far each line is from flat or upright; the median is the image's tilt
    tilt = np.degrees(theta) % 90
    tilt = np.where(tilt > 45, tilt - 90, tilt)
    tilt = tilt[np.abs(tilt) < 5]                                 # ignore diagonals
    angles[name] = float(np.median(tilt)) if len(tilt) else 0.0

    # 3. Rotate around the centre, filling the new corners with white
    turn = cv2.getRotationMatrix2D((w / 2, h / 2), angles[name], 1.0)
    cv2.imwrite(str(out_dir / "straightened" / name), cv2.warpAffine(img, turn, (w, h), borderValue=255))
    print(f"{name}: rotated {angles[name]:+.2f} deg")

In [ ]:
# Compare before/after; the red guide lines are perfectly straight,
# so tilted walls show up as drifting away from them
import matplotlib.pyplot as plt

for name in filenames:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, folder, title in ((axes[0], out_dir, "Thresholded"),
                              (axes[1], out_dir / "straightened", f"Straightened ({angles[name]:+.2f}°)")):
        im = Image.open(folder / name)
        ax.imshow(im, cmap="gray")
        for frac in np.linspace(0.1, 0.9, 9):
            ax.axhline(frac * im.height, color="red", lw=0.5)
            ax.axvline(frac * im.width, color="red", lw=0.5)
        ax.set_title(f"{title}: {name}")
        ax.axis("off")
    plt.show()